# Image-noise tuner

Tune the magnitudes in `training/image_noise.py` **by eye**, on a live game
frame. Needs pygame + PIL + numpy (the standard env) but no GPU, no NAMS, no
model.

- **Sliders** set the MAXIMUM of each magnitude range (the minimum keeps the
  production min/max ratio; the exact range in use is printed under the
  image): rectangle-tint opacity, whole-image color-drift opacity, speckle
  sigma.
- **Strength** previews the two production presets: `inference` (0.5, what
  the player sees during datagen) and `training` (1.0, what training sees) —
  every magnitude scales by it.
- **Regenerate** renders a brand-new random board AND redraws every random
  variable (patch positions/colors, drift color, speckle pattern, crop,
  jitter, blur-vs-JPEG coin). Moving a slider does NOT redraw anything:
  the same frozen scene is re-rendered at the new magnitude — that's a
  guarantee of `noise_image` itself (all random draws are unconditional,
  magnitudes only scale them).

The other noise sources (crop, brightness/contrast/color jitter, additive
gaussian, blur/JPEG) stay at their production magnitudes so what you see is
the full pipeline. When you like what you see, report the three printed
ranges and they become `_PATCH_ALPHA` / `_DRIFT_ALPHA` / `_SPECKLE_SIGMA`
in `training/image_noise.py`.

In [1]:
from __future__ import annotations

import io
import os
import random
import sys

# Run from the repo root so `agent`/`training`/`game` import (same as the
# other notebooks).
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
from PIL import Image

from agent.game_io import new_bare_game, render_frame_array
from training.image_noise import (
    _DRIFT_ALPHA,
    _PATCH_ALPHA,
    _SPECKLE_SIGMA,
    INFERENCE_STRENGTH,
    TRAINING_STRENGTH,
    noise_image,
)


def fresh_board() -> Image.Image:
    """A brand-new random bare board as a PIL image (headless render)."""
    arr = render_frame_array(new_bare_game())
    # pygame surfarray is (width, height, 3); PIL wants (height, width, 3).
    return Image.fromarray(np.transpose(arr, (1, 0, 2)), "RGB")


print("defaults:",
      f"_PATCH_ALPHA={_PATCH_ALPHA}",
      f"_DRIFT_ALPHA={_DRIFT_ALPHA}",
      f"_SPECKLE_SIGMA={_SPECKLE_SIGMA}")

/venv/main/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html
defaults: _PATCH_ALPHA=(0.12, 0.3) _DRIFT_ALPHA=(0.04, 0.1) _SPECKLE_SIGMA=(0.01, 0.04)


In [2]:
import ipywidgets as widgets

# Slider = the MAX of the range; the min keeps the production min/max ratio.
_RATIO = {
    "patch": _PATCH_ALPHA[0] / _PATCH_ALPHA[1],       # 0.4
    "drift": _DRIFT_ALPHA[0] / _DRIFT_ALPHA[1],       # 0.4
    "speckle": _SPECKLE_SIGMA[0] / _SPECKLE_SIGMA[1], # 0.25
}


def _range(kind: str, mx: float) -> tuple[float, float]:
    return (mx * _RATIO[kind], mx)


patch_sl = widgets.FloatSlider(
    value=_PATCH_ALPHA[1], min=0.0, max=0.8, step=0.01,
    description="rect alpha:", readout_format=".2f",
    layout=widgets.Layout(width="450px"),
)
drift_sl = widgets.FloatSlider(
    value=_DRIFT_ALPHA[1], min=0.0, max=0.5, step=0.01,
    description="drift alpha:", readout_format=".2f",
    layout=widgets.Layout(width="450px"),
)
speckle_sl = widgets.FloatSlider(
    value=_SPECKLE_SIGMA[1], min=0.0, max=0.15, step=0.005,
    description="speckle:", readout_format=".3f",
    layout=widgets.Layout(width="450px"),
)
strength_dd = widgets.Dropdown(
    options=[("inference (0.5)", INFERENCE_STRENGTH),
             ("training (1.0)", TRAINING_STRENGTH)],
    value=INFERENCE_STRENGTH, description="Strength:",
)
regen_btn = widgets.Button(description="Regenerate", button_style="warning")
img_w = widgets.Image(format="png", layout=widgets.Layout(width="560px"))
caption = widgets.HTML()

# The frozen scene: one board + one noise seed. Sliders re-render it;
# Regenerate replaces both.
_state = {"board": fresh_board(), "seed": random.getrandbits(32)}


def _render(*_):
    ranges = {
        "patch": _range("patch", patch_sl.value),
        "drift": _range("drift", drift_sl.value),
        "speckle": _range("speckle", speckle_sl.value),
    }
    # Fresh Random from the FROZEN seed: same crop, same patches, same
    # speckle pattern -- only the magnitudes move with the sliders.
    noised = noise_image(
        _state["board"], random.Random(_state["seed"]),
        strength=strength_dd.value,
        patch_alpha=ranges["patch"],
        drift_alpha=ranges["drift"],
        speckle_sigma=ranges["speckle"],
    )
    buf = io.BytesIO()
    noised.save(buf, format="PNG")
    img_w.value = buf.getvalue()
    caption.value = (
        "<code>"
        f"_PATCH_ALPHA = ({ranges['patch'][0]:.3f}, {ranges['patch'][1]:.3f})"
        f"<br>_DRIFT_ALPHA = ({ranges['drift'][0]:.3f}, {ranges['drift'][1]:.3f})"
        f"<br>_SPECKLE_SIGMA = ({ranges['speckle'][0]:.3f}, {ranges['speckle'][1]:.3f})"
        f"<br>strength = {strength_dd.value}"
        "</code>"
    )


def _regen(_):
    _state["board"] = fresh_board()
    _state["seed"] = random.getrandbits(32)
    _render()


for sl in (patch_sl, drift_sl, speckle_sl):
    sl.observe(_render, names="value")
strength_dd.observe(_render, names="value")
regen_btn.on_click(_regen)

display(widgets.VBox([
    widgets.HBox([strength_dd, regen_btn]),
    patch_sl,
    drift_sl,
    speckle_sl,
    img_w,
    caption,
]))
_render()